# Sionna 0.19 Scene Builder

This notebook is split into two independent sections:

**Section A — OSM Scene Builder** (cells 0–9): Run top-to-bottom to build a radio scene from OpenStreetMap data and elevation tiles, then verify and visualise it.

**Section B — Blender Scene Conversion** (cells B1–B3): Standalone converters for importing Blender-exported XML scenes into Sionna 0.19 or Sionna 2.0 format. Run these independently; they do not depend on Section A.


In [ ]:
# ============================================================
# CELL 0 — CONFIG  (edit this cell only)
# ============================================================
import os

# ── Scene bbox (WGS84) ────────────────────────────────────────────────────
# Tight bbox: TX + first 1200 RX (915.95 MHz) + 900m margin → 10.1×7.3 km = 74 km²
SCENE_WEST   = -1.260093
SCENE_EAST   = -1.129307
SCENE_SOUTH  =  52.945798
SCENE_NORTH  =  52.998702

# ── Output directory ─────────────────────────────────────────────────────
BASE_DIR  = os.path.expanduser('~/Documents/FYP2026/nottingham900')
SCENE_DIR = os.path.join(BASE_DIR, 'scene')
MESH_DIR  = os.path.join(SCENE_DIR, 'meshes')
os.makedirs(MESH_DIR, exist_ok=True)

# ── Coordinate system ────────────────────────────────────────────────────
UTM_EPSG  = 32630   # UTM zone 30N (covers UK)

# ── AWS tile zoom level ──────────────────────────────────────────────────
# z=14 → ~2.4 m/px at UK latitude (512×512 px tiles)
# z=13 → ~4.8 m/px  (faster download, less detail)
TILE_ZOOM  = 14

# ── Terrain mode ─────────────────────────────────────────────────────────────
# FLAT_TERRAIN = True  → fast flat z=0 plane, skip CELL 2 and CELL 2b
# FLAT_TERRAIN = False → real terrain from EA LiDAR DTM (auto-downloaded in CELL 2b)
#                        or AWS DEM tiles (CELL 2) — EA LiDAR recommended for UK
FLAT_TERRAIN = True   # ← change to False for realistic terrain (run CELL 2b first)

# ── Terrain source (only used when FLAT_TERRAIN=False) ────────────────────────
# 'ea_lidar' = Environment Agency 1m DTM (auto-download, UK only, most accurate)
# 'aws_dem'  = AWS elevation tiles 2.4m DSM (global, less accurate)
TERRAIN_SOURCE = 'ea_lidar'

# EA LiDAR output path (auto-set, edit only if needed)
EA_DTM_TIFF = os.path.join(BASE_DIR, 'ea_dtm_1m.tif')

# ── Terrain mesh resolution ───────────────────────────────────────────────
# Number of grid points per axis for the terrain PLY.
# 500 → 500×500 = 250k verts, ~500k triangles (~10 MB PLY) — recommended
# 200 → 200×200 = 40k  verts, ~80k triangles  (~1.5 MB PLY) — fast
TERRAIN_GRID_N = 500

# ── Building parameters ───────────────────────────────────────────────────
MIN_BUILDING_AREA_M2  = 30.0   # skip footprints smaller than this
CITY_MIN_HEIGHT_M     =  2.0   # clamp building height to at least this
CITY_MAX_HEIGHT_M     = 40.0   # clamp building height to at most this
HEIGHT_PER_LEVEL_M    =  3.5   # used when only building:levels tag present
DEFAULT_HEIGHT_M      =  8.0   # fallback when no height/levels tag

# ── OSM extra features ────────────────────────────────────────────────────
INCLUDE_ROADS       = True
INCLUDE_WATER       = True
INCLUDE_VEGETATION  = False   # polygon veg — adds clutter for macro-cell sim

# ── Exclude small/irrelevant building types ───────────────────────────────
EXCLUDE_BUILDING_TYPES = {
    'garage','garages','carport','shed','hut','roof','canopy',
    'kiosk','bicycle_parking','service','greenhouse','barn',
    'stable','sty','storage_tank','container','tent',
    'grandstand','shelter','utility','gatehouse',
}

# ── TX / RX parameters (915 MHz Ofcom drive-test) ────────────────────────
# From nottingham915.csv header:
#   Amp power=50.3 dBm, cable loss=1.3 dB  → TX_CONDUCTED_DBM = 49.0
#   TX antenna gain = 1.3 dBi               → EIRP = 50.3 dBm ≈ 55.1 per CSV
#   RX antenna gain = -1 dBi, cable = 0.2 dB, splitter = 6.1 dB
#                     LNA = 0 dB, BPF = 0.5 dB → RX_EXTRA_GAIN_DB = -7.8
TX_AGL_M             = 17.0    # TX antenna height above ground (m)
RX_AGL_M             =  1.5   # RX antenna height above ground (m)
TX_CONDUCTED_DBM     = 49.0   # dBm (amp output minus cable loss)
TX_ANTENNA_GAIN_DBI  =  1.3   # dBi (collinear omni, per CSV)
RX_EXTRA_GAIN_DB     = -7.8   # dB  (RX system gain: antenna + cable + filters)
SITE_CORRECTION_DB   =  0.0   # dB  (optional calibration offset, 0 = off)

# ── Antenna pattern ───────────────────────────────────────────────────────
# 'donut'  → half-wave dipole donut (max at horizon, null at zenith/nadir)
#            scaled to TX_ANTENNA_GAIN_DBI — matches Ofcom omni mast antenna
# 'iso'    → isotropic 0 dBi (uniform sphere) — safe fallback
ANTENNA_PATTERN = 'donut'

# ── Frequency ─────────────────────────────────────────────────────────────
FREQUENCY_HZ = 915.95e6   # Hz  (Ofcom 900 MHz band drive-test)

print('Config loaded.')
print(f'  Scene bbox  : lon [{SCENE_WEST}, {SCENE_EAST}]')
print(f'                lat [{SCENE_SOUTH}, {SCENE_NORTH}]')
print(f'  Output      : {SCENE_DIR}')
print(f'  Tile zoom   : {TILE_ZOOM}')
print(f'  Frequency   : {FREQUENCY_HZ/1e6:.2f} MHz')
print(f'  TX          : {TX_CONDUCTED_DBM:.1f} dBm conducted  +{TX_ANTENNA_GAIN_DBI:.1f} dBi  '
      f'EIRP={TX_CONDUCTED_DBM+TX_ANTENNA_GAIN_DBI:.1f} dBm  AGL={TX_AGL_M:.1f} m')
print(f'  RX          : system gain {RX_EXTRA_GAIN_DB:.1f} dB  AGL={RX_AGL_M:.1f} m')
print(f'  Pattern     : {ANTENNA_PATTERN}')

In [ ]:
# ============================================================
# CELL 1 — IMPORTS & DEPENDENCIES
# ============================================================
import os, math, time, json, struct, warnings
import numpy as np
import requests
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyproj import Transformer
import shapely.geometry as sg
import shapely.ops as so
from shapely.geometry import Polygon, MultiPolygon, box

# Optional: rasterio for GeoTIFF reading
try:
    import rasterio
    from rasterio.transform import from_bounds
    _HAS_RASTERIO = True
except ImportError:
    _HAS_RASTERIO = False
    print('⚠  rasterio not found — falling back to PIL for GeoTIFF tiles')

# PIL for fallback
try:
    from PIL import Image
    _HAS_PIL = True
except ImportError:
    _HAS_PIL = False

# osmnx for OSM data
try:
    import osmnx as ox
    ox.settings.use_cache = True
    ox.settings.log_console = False
    _HAS_OSMNX = True
except ImportError:
    _HAS_OSMNX = False
    print('⚠  osmnx not found — install with: pip install osmnx')

# trimesh for PLY export
try:
    import trimesh
    _HAS_TRIMESH = True
except ImportError:
    _HAS_TRIMESH = False
    print('⚠  trimesh not found — install with: pip install trimesh')

# Coordinate transformers
to_utm   = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
to_wgs84 = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)

# Scene bbox in UTM
sw_utm = to_utm.transform(SCENE_WEST,  SCENE_SOUTH)
ne_utm = to_utm.transform(SCENE_EAST,  SCENE_NORTH)
center_utm = ((sw_utm[0]+ne_utm[0])/2, (sw_utm[1]+ne_utm[1])/2)
center_lon, center_lat = to_wgs84.transform(*center_utm)

print(f'UTM SW : {sw_utm[0]:.1f}, {sw_utm[1]:.1f}')
print(f'UTM NE : {ne_utm[0]:.1f}, {ne_utm[1]:.1f}')
print(f'Center : ({center_lon:.5f}, {center_lat:.5f})')
print(f'Size   : {(ne_utm[0]-sw_utm[0])/1000:.2f} km × {(ne_utm[1]-sw_utm[1])/1000:.2f} km')

In [ ]:
# ============================================================
# CELL 2 — AWS ELEVATION TILES → HEIGHTMAP
# ============================================================
# Downloads GeoTIFF tiles from the public AWS elevation-tiles-prod bucket
# (same source used by Mapzen/Terrarium, Cesium, sionna-large-radio-maps).
# No credentials needed — anonymous public access.
# URL: s3://elevation-tiles-prod/geotiff/{z}/{x}/{y}.tif
# Values are direct metres ASL (float32 GeoTIFF, no conversion formula).

AWS_BASE = 'https://s3.amazonaws.com/elevation-tiles-prod/geotiff'

def _lon2tile(lon, z):
    return int(math.floor((lon + 180) / 360 * 2**z))

def _lat2tile(lat, z):
    lat_r = math.radians(lat)
    return int(math.floor((1 - math.log(math.tan(lat_r) + 1/math.cos(lat_r)) / math.pi) / 2 * 2**z))

def _tile2lon(x, z):
    return x / 2**z * 360 - 180

def _tile2lat(y, z):
    n = math.pi - 2 * math.pi * y / 2**z
    return math.degrees(math.atan(math.sinh(n)))

def _download_tile(z, x, y, retries=3):
    url = f'{AWS_BASE}/{z}/{x}/{y}.tif'
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            buf = BytesIO(r.content)
            if _HAS_RASTERIO:
                with rasterio.open(buf) as ds:
                    data = ds.read(1).astype(np.float32)
            elif _HAS_PIL:
                # PIL may not read float GeoTIFF correctly — warn
                img = Image.open(buf)
                data = np.array(img, dtype=np.float32)
            else:
                raise RuntimeError('Need rasterio or PIL to read GeoTIFF tiles')
            # Resample to 512×512 if needed
            if data.shape != (512, 512):
                from scipy.ndimage import zoom as nd_zoom
                fx = 512 / data.shape[0]; fy = 512 / data.shape[1]
                data = nd_zoom(data, (fx, fy), order=1).astype(np.float32)
            return data
        except Exception as e:
            if attempt == retries - 1:
                print(f'  ⚠ tile {z}/{x}/{y} failed: {e} — using zeros')
                return np.zeros((512, 512), dtype=np.float32)
            time.sleep(2 ** attempt)

# Compute tile range for scene bbox
x0 = _lon2tile(SCENE_WEST,  TILE_ZOOM)
x1 = _lon2tile(SCENE_EAST,  TILE_ZOOM)
y0 = _lat2tile(SCENE_NORTH, TILE_ZOOM)  # north = smaller y in tile coords
y1 = _lat2tile(SCENE_SOUTH, TILE_ZOOM)
n_cols = x1 - x0 + 1
n_rows = y1 - y0 + 1
print(f'Tile range : x=[{x0},{x1}] y=[{y0},{y1}]  →  {n_cols}×{n_rows} = {n_cols*n_rows} tiles')

# Download in parallel
tile_mosaic = np.zeros((n_rows * 512, n_cols * 512), dtype=np.float32)
jobs = [(TILE_ZOOM, x0+col, y0+row, col, row)
        for row in range(n_rows) for col in range(n_cols)]

print(f'Downloading {len(jobs)} tiles ...')
t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {ex.submit(_download_tile, z, x, y): (col, row)
               for z, x, y, col, row in jobs}
    for i, fut in enumerate(as_completed(futures)):
        col, row = futures[fut]
        tile_mosaic[row*512:(row+1)*512, col*512:(col+1)*512] = fut.result()
        if (i+1) % max(1, len(jobs)//4) == 0:
            print(f'  {i+1}/{len(jobs)} tiles done')

print(f'Download done in {time.time()-t0:.1f}s')

# Extent of the mosaic in WGS84
_map_min_lon = _tile2lon(x0,   TILE_ZOOM)
_map_max_lon = _tile2lon(x1+1, TILE_ZOOM)
_map_max_lat = _tile2lat(y0,   TILE_ZOOM)   # y0 is north
_map_min_lat = _tile2lat(y1+1, TILE_ZOOM)   # y1+1 is south
_mosaic_h, _mosaic_w = tile_mosaic.shape

print(f'Mosaic size : {_mosaic_w}×{_mosaic_h} px')
print(f'Mosaic lon  : [{_map_min_lon:.5f}, {_map_max_lon:.5f}]')
print(f'Mosaic lat  : [{_map_min_lat:.5f}, {_map_max_lat:.5f}]')
print(f'Elevation   : [{tile_mosaic.min():.1f}, {tile_mosaic.max():.1f}] m ASL')

def height_from_wgs84(lon, lat):
    """Bilinear interpolation from mosaic. Returns elevation in metres ASL."""
    u = (lon - _map_min_lon) / (_map_max_lon - _map_min_lon) * (_mosaic_w - 1)
    v = (1 - (lat - _map_min_lat) / (_map_max_lat - _map_min_lat)) * (_mosaic_h - 1)
    u = np.clip(u, 0, _mosaic_w - 1)
    v = np.clip(v, 0, _mosaic_h - 1)
    x0i, y0i = int(np.floor(u)), int(np.floor(v))
    x1i, y1i = min(x0i+1, _mosaic_w-1), min(y0i+1, _mosaic_h-1)
    fx, fy = u - x0i, v - y0i
    h = (tile_mosaic[y0i, x0i] * (1-fx) * (1-fy)
       + tile_mosaic[y0i, x1i] *    fx  * (1-fy)
       + tile_mosaic[y1i, x0i] * (1-fx) *    fy
       + tile_mosaic[y1i, x1i] *    fx  *    fy)
    return float(h)

def height_from_utm(easting, northing):
    """Height at UTM coordinates."""
    lon, lat = to_wgs84.transform(easting, northing)
    return height_from_wgs84(lon, lat)

# Scene centre elevation = local z=0 reference
origin_elev_asl = height_from_wgs84(center_lon, center_lat)
print(f'\nScene centre elevation : {origin_elev_asl:.2f} m ASL  (= local z=0)')

def local_z(lon, lat):
    """Returns local z in metres relative to scene centre elevation."""
    return height_from_wgs84(lon, lat) - origin_elev_asl

In [ ]:
# ============================================================
# CELL 2b — EA LiDAR DTM Auto-Download  (skip if FLAT_TERRAIN=True)
# ============================================================
# Downloads Environment Agency 1m Composite DTM for scene bbox.
# Source: EA WCS service (free, no credentials, England only).
# CRS: EPSG:27700 (British National Grid) for request, output GeoTIFF.
# Skip this cell entirely when FLAT_TERRAIN=True.
# ============================================================

if globals().get('FLAT_TERRAIN', True):
    print('FLAT_TERRAIN=True — skipping EA LiDAR download.')
    print('Set FLAT_TERRAIN=False in CELL 0 and re-run this cell for real terrain.')
else:
    import requests, os
    from pyproj import Transformer

    print('=' * 60)
    print('CELL 2b — EA LiDAR DTM Download (1m resolution)')
    print('=' * 60)

    if os.path.exists(EA_DTM_TIFF):
        print(f'Already downloaded: {EA_DTM_TIFF}')
        print('Delete the file and re-run to force re-download.')
    else:
        # Convert scene bbox WGS84 → BNG (EPSG:27700) for EA WCS request
        _wgs_to_bng = Transformer.from_crs('EPSG:4326', 'EPSG:27700', always_xy=True)
        _e_min, _n_min = _wgs_to_bng.transform(SCENE_WEST,  SCENE_SOUTH)
        _e_max, _n_max = _wgs_to_bng.transform(SCENE_EAST,  SCENE_NORTH)

        # Add 200m buffer so terrain edge doesn't clip buildings
        _buf = 200
        _e_min -= _buf; _n_min -= _buf
        _e_max += _buf; _n_max += _buf

        print(f'BNG bbox: E[{_e_min:.0f}, {_e_max:.0f}]  N[{_n_min:.0f}, {_n_max:.0f}]')
        print(f'Area    : {(_e_max-_e_min)/1000:.1f} km × {(_n_max-_n_min)/1000:.1f} km')

        # EA WCS endpoint — Composite DTM 1m
        _WCS_URL = (
            'https://environment.data.gov.uk/spatialdata/lidar-composite-dtm-1m/wcs'
            '?SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage'
            '&COVERAGEID=LIDAR_Composite_DTM_1m'
            f'&SUBSET=E,http://www.opengis.net/def/crs/EPSG/0/27700({_e_min:.0f},{_e_max:.0f})'
            f'&SUBSET=N,http://www.opengis.net/def/crs/EPSG/0/27700({_n_min:.0f},{_n_max:.0f})'
            '&FORMAT=image/tiff'
        )

        print(f'Downloading EA LiDAR DTM ...')
        print(f'URL: {_WCS_URL[:100]}...')

        try:
            _r = requests.get(_WCS_URL, timeout=120, stream=True)
            _r.raise_for_status()
            os.makedirs(os.path.dirname(EA_DTM_TIFF), exist_ok=True)
            _total = 0
            with open(EA_DTM_TIFF, 'wb') as _f:
                for _chunk in _r.iter_content(chunk_size=1024*1024):
                    _f.write(_chunk)
                    _total += len(_chunk)
                    print(f'  {_total//1024//1024} MB downloaded ...', end='\r')
            print(f'\nSaved: {EA_DTM_TIFF}  ({os.path.getsize(EA_DTM_TIFF)//1024//1024} MB)')

            # Quick verify with rasterio
            if _HAS_RASTERIO:
                import rasterio as _rio
                with _rio.open(EA_DTM_TIFF) as _ds:
                    print(f'CRS     : {_ds.crs}')
                    print(f'Shape   : {_ds.height} × {_ds.width} px')
                    print(f'Res     : {_ds.res[0]:.1f} m/px')
                    _data = _ds.read(1)
                    print(f'Z range : {float(_data.min()):.1f} – {float(_data.max()):.1f} m ASL')
            print('\n✓ EA LiDAR DTM ready. Now run CELL 3 to build terrain mesh.')

        except requests.exceptions.HTTPError as _e:
            print(f'HTTP error: {_e}')
            print('The EA WCS may be temporarily unavailable. Try again later.')
            print('Alternative: download manually from:')
            print('  https://environment.data.gov.uk/DefraDataDownload/?Mode=survey')
            print('  Select: LIDAR Composite DTM → 1m → your area → GeoTIFF')
            print(f'  Save to: {EA_DTM_TIFF}')
        except Exception as _e:
            print(f'Download failed: {_e}')
            print(f'Manual download URL:')
            print('  https://environment.data.gov.uk/DefraDataDownload/?Mode=survey')


In [ ]:
# ============================================================
# CELL 3 — BUILD TERRAIN PLY
# ============================================================
# FLAT_TERRAIN=True  : flat z=0 plane — fast, no DEM needed
# FLAT_TERRAIN=False : DEM elevation from AWS tiles (run CELL 2 first)
# ============================================================
import struct, os
import numpy as np

N = TERRAIN_GRID_N

if FLAT_TERRAIN:
    print(f'Building FLAT terrain mesh: {N}×{N} grid ...')
    x_span = (SCENE_EAST  - SCENE_WEST)  * 111000 * np.cos(np.radians((SCENE_SOUTH+SCENE_NORTH)/2))
    y_span = (SCENE_NORTH - SCENE_SOUTH) * 111000
    xs = np.linspace(-x_span/2, x_span/2, N, dtype=np.float32)
    ys = np.linspace(-y_span/2, y_span/2, N, dtype=np.float32)
    XX, YY = np.meshgrid(xs, ys, indexing='ij')
    ZZ = np.zeros((N, N), dtype=np.float32)
    origin_elev_asl = 0.0
    def local_z(lon, lat): return 0.0
else:
    _src = globals().get('TERRAIN_SOURCE', 'aws_dem')
    if _src == 'ea_lidar' and os.path.exists(globals().get('EA_DTM_TIFF','')):
        print(f'Building terrain mesh from EA LiDAR DTM: {N}×{N} grid ...')
        import rasterio as _rio
        from pyproj import Transformer as _Tr
        _bng_to_utm = _Tr.from_crs('EPSG:27700', f'EPSG:{UTM_EPSG}', always_xy=True)
        _utm_to_bng = _Tr.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:27700', always_xy=True)
        _ds = _rio.open(EA_DTM_TIFF)
        _dem_data = _ds.read(1).astype(np.float32)
        _dem_tf   = _ds.transform
        _dem_nd   = _ds.nodata

        # scene origin elevation from EA DTM
        _ox_bng, _oy_bng = _utm_to_bng.transform(center_utm[0], center_utm[1])
        _oc, _or = ~_dem_tf * (_ox_bng, _oy_bng)
        _or, _oc = int(_or), int(_oc)
        if 0 <= _or < _dem_data.shape[0] and 0 <= _oc < _dem_data.shape[1]:
            origin_elev_asl = float(_dem_data[_or, _oc])
        else:
            origin_elev_asl = 0.0
        print(f'  Origin elevation: {origin_elev_asl:.1f} m ASL')

        def _ea_z(utm_x, utm_y):
            bx, by = _utm_to_bng.transform(utm_x, utm_y)
            col_f, row_f = ~_dem_tf * (bx, by)
            r, c = int(row_f), int(col_f)
            H, W = _dem_data.shape
            if 0 <= r < H-1 and 0 <= c < W-1:
                dr, dc = row_f-r, col_f-c
                z = ((1-dr)*(1-dc)*_dem_data[r,c] + (1-dr)*dc*_dem_data[r,c+1] +
                     dr*(1-dc)*_dem_data[r+1,c] + dr*dc*_dem_data[r+1,c+1])
                if _dem_nd is None or not np.isclose(float(z), _dem_nd):
                    return float(z)
            return origin_elev_asl

        x_span = (SCENE_EAST-SCENE_WEST)*111000*np.cos(np.radians((SCENE_SOUTH+SCENE_NORTH)/2))
        y_span = (SCENE_NORTH-SCENE_SOUTH)*111000
        xs = np.linspace(-x_span/2, x_span/2, N, dtype=np.float32)
        ys = np.linspace(-y_span/2, y_span/2, N, dtype=np.float32)
        XX, YY = np.meshgrid(xs, ys, indexing='ij')
        ZZ = np.zeros((N, N), dtype=np.float32)
        import time as _t; t0 = _t.time()
        for i in range(N):
            for j in range(N):
                ZZ[i,j] = _ea_z(center_utm[0]+XX[i,j], center_utm[1]+YY[i,j]) - origin_elev_asl
            if (i+1) % max(1,N//5)==0:
                print(f'  row {i+1}/{N}  ({_t.time()-t0:.0f}s)')
        def local_z(lon, lat):
            ux,uy = to_utm.transform(lon,lat)
            return _ea_z(ux,uy) - origin_elev_asl
    else:
        print(f'Building terrain mesh from AWS DEM: {N}×{N} grid ...')
        x_span = ne_utm[0] - sw_utm[0]
        y_span = ne_utm[1] - sw_utm[1]
        xs = np.linspace(-x_span/2, x_span/2, N, dtype=np.float32)
        ys = np.linspace(-y_span/2, y_span/2, N, dtype=np.float32)
        XX, YY = np.meshgrid(xs, ys, indexing='ij')
        ZZ = np.zeros((N, N), dtype=np.float32)
        import time as _t; t0 = _t.time()
        for i in range(N):
            for j in range(N):
                utm_x = center_utm[0] + XX[i, j]
                utm_y = center_utm[1] + YY[i, j]
                ZZ[i, j] = height_from_utm(utm_x, utm_y) - origin_elev_asl
            if (i+1) % max(1, N//5) == 0:
                print(f'  row {i+1}/{N}  ({_t.time()-t0:.0f}s)')

print(f'Terrain Z range: [{ZZ.min():.1f}, {ZZ.max():.1f}] m')

verts = np.stack([XX.ravel(), YY.ravel(), ZZ.ravel()], axis=1).astype(np.float32)
faces = []
for i in range(N-1):
    for j in range(N-1):
        a = i*N+j; b = a+1; c = a+N; d = c+1
        faces.append([a,b,d]); faces.append([a,d,c])
faces = np.array(faces, dtype=np.int32)

# Write binary PLY
ply_path = os.path.join(MESH_DIR, 'terrain.ply')
with open(ply_path, 'wb') as f:
    hdr = (f'ply\nformat binary_little_endian 1.0\n'
           f'element vertex {len(verts)}\nproperty float x\nproperty float y\nproperty float z\n'
           f'element face {len(faces)}\nproperty list uchar int vertex_indices\nend_header\n')
    f.write(hdr.encode())
    f.write(verts.tobytes())
    for fc in faces:
        f.write(struct.pack('<B3i', 3, *fc))

kb = os.path.getsize(ply_path)//1024
print(f'terrain.ply: {len(verts):,} verts  {len(faces):,} faces  {kb} KB  → {ply_path}')

# Save origin elevation for main notebook compatibility
import json as _json
_elev_path = os.path.join(SCENE_DIR, 'origin_elev2.json')
_json.dump({'origin_elev_asl_m': float(origin_elev_asl), 'flat_terrain': FLAT_TERRAIN}, open(_elev_path, 'w'))
print(f'origin_elev2.json: {_elev_path}')


In [ ]:
# ============================================================
# CELL 4 — OSM BUILDINGS → BUILDING PLYS
# ============================================================

assert _HAS_OSMNX, 'osmnx required: pip install osmnx'

import osmnx as _ox_ver
_ox_version = tuple(int(x) for x in _ox_ver.__version__.split('.')[:2])

print(f'Downloading OSM buildings (osmnx {_ox_ver.__version__}) ...')
t0 = time.time()
try:
    if _ox_version >= (2, 0):
        gdf_bld = ox.features_from_bbox(
            bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH),
            tags={'building': True}
        )
    elif _ox_version >= (1, 3):
        gdf_bld = ox.features_from_bbox(
            bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST),
            tags={'building': True}
        )
    else:
        gdf_bld = ox.features_from_bbox(
            north=SCENE_NORTH, south=SCENE_SOUTH,
            east=SCENE_EAST,   west=SCENE_WEST,
            tags={'building': True}
        )
except Exception as e:
    print(f'osmnx error: {e}')
    raise
print(f'  {len(gdf_bld)} raw building features  ({time.time()-t0:.1f}s)')

# ── Material helpers ───────────────────────────────────────────────────────
def _bld_mat(row):
    mat  = str(row.get('building:material', '')).lower()
    fmat = str(row.get('building:facade:material', '')).lower()
    tag  = str(row.get('building', '')).lower()
    amen = str(row.get('amenity', '')).lower()
    shop = str(row.get('shop', '')).lower()
    off  = str(row.get('office', '')).lower()
    if 'glass' in mat or 'glass' in fmat:                return 'itu_glass'
    if 'wood'  in mat or 'timber' in mat:                return 'itu_wood'
    if 'wood'  in fmat or 'timber' in fmat:              return 'itu_wood'
    if 'brick' in mat or 'brick' in fmat:                return 'itu_brick'
    if 'stone' in mat or 'stone' in fmat:                return 'itu_brick'
    if tag in ('greenhouse','glasshouse'):               return 'itu_glass'
    if amen in ('shopping_centre','mall'):               return 'itu_glass'
    if shop in ('mall','supermarket','department_store'):return 'itu_glass'
    if off:                                              return 'itu_glass'
    if tag in ('residential','house','detached','semidetached_house',
               'semi_detached','terrace','terrace_house','bungalow',
               'farm','farmhouse','dormitory','apartments','block'):
        return 'itu_brick'
    if tag in ('industrial','warehouse','factory','shed',
               'storage_tank','silo','barn'):            return 'itu_concrete'
    if tag in ('retail','commercial','supermarket','kiosk'): return 'itu_glass'
    if tag in ('cathedral','church','chapel','mosque','temple'): return 'itu_brick'
    if tag in ('school','university','hospital','civic','public'): return 'itu_concrete'
    return 'itu_brick'

def _roof_mat(row):
    roof_tag = str(row.get('roof:material', '')).lower()
    btag     = str(row.get('building', '')).lower()
    if any(k in roof_tag for k in ['metal','steel','zinc','aluminium','copper','tin']):
        return 'itu_metal'
    if 'glass' in roof_tag:                              return 'itu_glass'
    if any(k in roof_tag for k in ['wood','timber','thatch']): return 'itu_wood'
    if any(k in roof_tag for k in ['tile','concrete','slate','terracotta']):
        return 'itu_concrete'
    if btag in ('industrial','warehouse','factory','shed','barn',
                'retail','supermarket','commercial','garage','garages'):
        return 'itu_metal'
    return 'itu_concrete'

def _bld_height(row):
    try:
        h = float(str(row.get('height','0')).replace('m','').strip())
        if h > 1: return np.clip(h, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M)
    except: pass
    try:
        lvl = float(str(row.get('building:levels','0')).strip())
        if lvl > 0: return np.clip(lvl * HEIGHT_PER_LEVEL_M, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M)
    except: pass
    return DEFAULT_HEIGHT_M

def _write_ply(verts, faces, path):
    verts = np.asarray(verts, dtype=np.float32)
    faces = np.asarray(faces, dtype=np.int32)
    if _HAS_TRIMESH:
        trimesh.Trimesh(vertices=verts, faces=faces, process=False).export(path)
        return
    with open(path, 'w') as f:
        f.write('ply\nformat ascii 1.0\n')
        f.write(f'element vertex {len(verts)}\n')
        f.write('property float x\nproperty float y\nproperty float z\n')
        f.write(f'element face {len(faces)}\n')
        f.write('property list uchar int vertex_indices\nend_header\n')
        for v in verts:   f.write(f'{v[0]:.4f} {v[1]:.4f} {v[2]:.4f}\n')
        for fc in faces:  f.write(f'3 {fc[0]} {fc[1]} {fc[2]}\n')

def _triangulate_roof(pts_2d, top_z):
    """
    Robust Delaunay roof triangulation via Shapely.
    Handles convex AND concave footprints — no degenerate triangles.
    """
    poly = sg.Polygon(pts_2d)
    if not poly.is_valid:
        poly = poly.buffer(0)   # auto-repair self-intersections
    if poly.is_empty or poly.area < 0.1:
        return None

    try:
        tris = so.triangulate(poly)
    except Exception:
        return None

    v_idx = {}
    roof_v = []
    roof_f = []

    for tri in tris:
        # Keep only triangles whose centroid lies inside the footprint
        if not poly.contains(tri.centroid):
            continue
        coords = list(tri.exterior.coords)[:3]
        face_idxs = []
        for cx, cy in coords:
            key = (round(cx, 3), round(cy, 3))
            if key not in v_idx:
                v_idx[key] = len(roof_v)
                roof_v.append([cx, cy, top_z])
            face_idxs.append(v_idx[key])
        # Skip zero-area triangles (cross product check)
        a = np.array(roof_v[face_idxs[1]][:2]) - np.array(roof_v[face_idxs[0]][:2])
        b = np.array(roof_v[face_idxs[2]][:2]) - np.array(roof_v[face_idxs[0]][:2])
        if abs(a[0]*b[1] - a[1]*b[0]) > 1e-6:
            roof_f.append(face_idxs)

    if not roof_v or not roof_f:
        return None
    return np.array(roof_v, np.float32), np.array(roof_f, np.int32)

def _extrude_building(poly_utm, base_z, height):
    pts = np.array(poly_utm, dtype=np.float32)
    if len(pts) < 3:
        return None
    if np.allclose(pts[0], pts[-1]):
        pts = pts[:-1]
    n = len(pts)
    top_z = base_z + height
    bot_z = base_z

    # ── Walls ──────────────────────────────────────────────────────────────
    wall_v, wall_f = [], []
    for i in range(n):
        j = (i+1) % n
        v_base = len(wall_v)
        wall_v += [
            [pts[i,0], pts[i,1], bot_z],
            [pts[j,0], pts[j,1], bot_z],
            [pts[j,0], pts[j,1], top_z],
            [pts[i,0], pts[i,1], top_z],
        ]
        wall_f += [
            [v_base,   v_base+1, v_base+2],
            [v_base,   v_base+2, v_base+3],
        ]

    # ── Roof — Shapely Delaunay (no degenerate triangles) ─────────────────
    roof_result = _triangulate_roof([(pts[i,0], pts[i,1]) for i in range(n)], top_z)
    if roof_result is None:
        return None   # skip buildings with unfixable geometry

    rv, rf = roof_result
    return (np.array(wall_v, np.float32), np.array(wall_f, np.int32), rv, rf)

# ── Process buildings — accumulate geometry per material ────────────────────
mat_geom = {}   # mat -> {'verts': [], 'faces': [], 'offset': 0}
n_ok = n_skip = 0
_n_total = len(gdf_bld)
t0 = time.time()

for idx, (oid, row) in enumerate(gdf_bld.iterrows()):
    geom = row.geometry
    if geom is None or geom.is_empty:
        n_skip += 1; continue

    if isinstance(geom, MultiPolygon):
        geom = max(geom.geoms, key=lambda g: g.area)
    if not isinstance(geom, Polygon):
        n_skip += 1; continue

    btag = str(row.get('building','')).lower()
    if btag in EXCLUDE_BUILDING_TYPES:
        n_skip += 1; continue

    coords_wgs = list(geom.exterior.coords)
    coords_utm = []
    for lon, lat in coords_wgs:
        ex, ny = to_utm.transform(lon, lat)
        coords_utm.append((ex - center_utm[0], ny - center_utm[1]))

    area_m2 = sg.Polygon(coords_utm).area
    if area_m2 < MIN_BUILDING_AREA_M2:
        n_skip += 1; continue

    c_lon, c_lat = geom.centroid.x, geom.centroid.y
    base_z_local = local_z(c_lon, c_lat)
    h = _bld_height(row)

    result = _extrude_building(coords_utm, base_z_local, h)
    if result is None:
        n_skip += 1; continue
    wv, wf, rv, rf = result

    w_mat = _bld_mat(row)
    r_mat = _roof_mat(row)

    for mat, verts, faces in [(w_mat, wv, wf), (r_mat, rv, rf)]:
        if mat not in mat_geom:
            mat_geom[mat] = {'verts': [], 'faces': [], 'offset': 0}
        g = mat_geom[mat]
        g['faces'].append(faces + g['offset'])
        g['verts'].append(verts)
        g['offset'] += len(verts)

    n_ok += 1
    if n_ok % 100 == 0:
        print(f'  {n_ok}/{_n_total} buildings  ({time.time()-t0:.0f}s elapsed) ...')

print(f'\nBuildings : {n_ok} exported, {n_skip} skipped')
print(f'Materials : {list(mat_geom.keys())}')

# ── Write one merged PLY per material ───────────────────────────────────────
mat_plys = {}
for mat, g in mat_geom.items():
    all_v = np.concatenate(g['verts'], axis=0).astype(np.float32)
    all_f = np.concatenate(g['faces'], axis=0).astype(np.int32)
    ply_name = f'bld_{mat}.ply'
    ply_path = os.path.join(MESH_DIR, ply_name)
    _write_ply(all_v, all_f, ply_path)
    mat_plys[mat] = [('meshes/' + ply_name, 'buildings')]
    print(f'  {mat}: {len(all_v)} verts, {len(all_f)} faces → {ply_name}')

print(f'\nPLY files written: {len(mat_plys)} (one per material)')


In [ ]:
# ============================================================
# CELL 5 — WRITE SCENE.XML  (Mitsuba 2.1.0 / Sionna 0.19)
# ============================================================
# ITU-R P.2040-2 material definitions as used in Sionna 0.19.
# Each shape references a PLY file and a material by name.

ITU_MATERIALS = {
    # name          : (relative_permittivity, conductivity_S_m)
    'itu_concrete'  : (5.31,  0.092),
    'itu_brick'     : (3.75,  0.038),
    'itu_glass'     : (6.27,  0.000),
    'itu_wood'      : (1.99,  0.000),
    'itu_metal'     : (1.00, 1.0e7 ),
    'itu_asphalt'   : (2.56,  0.000),
    'itu_wet_ground': (30.0,  0.020),
}
TERRAIN_MATERIAL = 'itu_wet_ground'

# Collect all materials actually used
used_mats = set(mat_plys.keys()) | {TERRAIN_MATERIAL}

lines = []
lines.append('<?xml version="1.0" encoding="utf-8"?>')
lines.append('<scene version="2.1.0">')
lines.append('')
lines.append('  <!-- ── ITU-R P.2040-2 Materials ────────────────────── -->')

for mat_name, (eps, sigma) in ITU_MATERIALS.items():
    if mat_name not in used_mats:
        continue
    lines.append(f'  <bsdf type="conductor" id="{mat_name}">')
    lines.append(f'    <float name="eta" value="{eps}"/>')
    lines.append(f'    <float name="k"   value="{sigma}"/>')
    lines.append(f'  </bsdf>')
    lines.append('')

lines.append('  <!-- ── Terrain ─────────────────────────────────────── -->')
lines.append('  <shape type="ply">')
lines.append('    <string name="filename" value="meshes/terrain.ply"/>')
lines.append('    <ref id="{mat}" name="bsdf"/>'.replace('{mat}', TERRAIN_MATERIAL))
lines.append('  </shape>')
lines.append('')

lines.append('  <!-- ── Buildings ───────────────────────────────────── -->')
for mat_name, ply_list in sorted(mat_plys.items()):
    for ply_path, role in ply_list:
        lines.append(f'  <shape type="ply">')
        lines.append(f'    <string name="filename" value="{ply_path}"/>')
        lines.append(f'    <ref id="{mat_name}" name="bsdf"/>')
        lines.append(f'  </shape>')

lines.append('')
lines.append('</scene>')

scene_xml = os.path.join(SCENE_DIR, 'scene.xml')
with open(scene_xml, 'w') as f:
    f.write('\n'.join(lines))

print(f'Wrote: {scene_xml}')
print(f'  Materials : {len(used_mats)}')
total_shapes = 1 + sum(len(v) for v in mat_plys.values())
print(f'  Shapes    : {total_shapes}  (1 terrain + {total_shapes-1} building parts)')

# Save scene metadata for main notebook
meta = {
    'scene_center_lon'  : center_lon,
    'scene_center_lat'  : center_lat,
    'origin_elev_asl_m' : origin_elev_asl,
    'utm_epsg'          : UTM_EPSG,
    'bbox'              : {'west': SCENE_WEST, 'east': SCENE_EAST,
                           'south': SCENE_SOUTH, 'north': SCENE_NORTH},
    'n_buildings'       : n_ok,
    'terrain_grid_n'    : TERRAIN_GRID_N,
    'tile_zoom'         : TILE_ZOOM,
}
params_json = os.path.join(BASE_DIR, 'scene_parameters.json')
with open(params_json, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'  Metadata  : {params_json}')

In [ ]:
# ============================================================
# CELL 6 — VERIFY SCENE
# ============================================================
# Quick sanity checks before handing the scene to the main notebook.

import glob as glob_mod

ply_files = sorted(glob_mod.glob(os.path.join(MESH_DIR, '*.ply')))
total_kb = sum(os.path.getsize(p) for p in ply_files) / 1024

print('=' * 60)
print('SCENE VERIFICATION')
print('=' * 60)
print(f'PLY files   : {len(ply_files)}')
print(f'Total size  : {total_kb/1024:.1f} MB')
print()

print(f'scene.xml   : {os.path.getsize(scene_xml)/1024:.0f} KB')

# Load with Mitsuba to verify (requires Sionna env)
try:
    import mitsuba as mi
    mi.set_variant('scalar_rgb')
    scene_mi = mi.load_file(scene_xml)
    bbox = scene_mi.bbox()
    print()
    print(f'Mitsuba load: OK')
    print(f'  BBox X    : [{float(bbox.min[0]):.1f}, {float(bbox.max[0]):.1f}] m')
    print(f'  BBox Y    : [{float(bbox.min[1]):.1f}, {float(bbox.max[1]):.1f}] m')
    print(f'  BBox Z    : [{float(bbox.min[2]):.1f}, {float(bbox.max[2]):.1f}] m')
except Exception as e:
    print(f'Mitsuba load: {e}')

print()
print('Scene metadata (scene_parameters.json):')
with open(params_json) as f:
    print(json.dumps(json.load(f), indent=2))

print()
print('DONE — scene ready for sionna019_main_simulation.ipynb')
print(f'Set BASE_DIR = "{BASE_DIR}" in Cell 0c of the main notebook.')

## CELL 7 — 2D Scene Map: OSM Buildings + Receiver Locations + RSSI Heatmap


In [ ]:
# ============================================================
# CELL 7 — 2D MAP: OSM BUILDINGS + TX/RX DOTS
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
import re, warnings

# ── Config — edit here ────────────────────────────────────────────────────
OFCOM_CSV   = os.path.join(BASE_DIR, 'nottingham915.csv')

# Number of RX points to plot (first N rows in CSV order, same as simulation).
# Set to None to plot all rows in the file.
NUM_RX_PLOT = 1200      # ← adjust to match NUM_RX in simulation notebook

SAVE_FIG_2D = True
FIG_DPI_2D  = 150

# ── Auto-detect header row + parse CSV ───────────────────────────────────
_tx_lat = _tx_lon = None
_rx_rows = []

if not os.path.exists(OFCOM_CSV):
    print(f'CSV not found: {OFCOM_CSV}')
else:
    # Read TX site coordinates from metadata lines
    with open(OFCOM_CSV, 'r', encoding='utf-8', errors='replace') as _f:
        _all_lines = _f.readlines()

    for _line in _all_lines[:40]:
        if 'latitude'  in _line.lower() and _tx_lat is None:
            _m = re.search(r'([-+]?\d+\.\d+)', _line)
            if _m: _tx_lat = float(_m.group(1))
        if 'longitude' in _line.lower() and _tx_lon is None:
            _m = re.search(r'([-+]?\d+\.\d+)', _line)
            if _m: _tx_lon = float(_m.group(1))

    # Find header row: first line containing both 'Latitude' and 'Longitude'
    _hdr_idx = next(
        (i for i, l in enumerate(_all_lines) if 'Latitude' in l and 'Longitude' in l),
        None
    )

    if _hdr_idx is None:
        print('WARNING: could not find header row in CSV')
    else:
        _df = pd.read_csv(OFCOM_CSV, skiprows=_hdr_idx, low_memory=False)

        # Flexible column matching
        def _fcol(df, *kws):
            for c in df.columns:
                if all(k.lower() in c.strip().lower() for k in kws):
                    return c
            return None

        _lat_col  = _fcol(_df, 'latitude')
        _lon_col  = _fcol(_df, 'longitude')
        _rssi_col = _fcol(_df, 'measurement') or _fcol(_df, 'dbm')

        if _lat_col and _lon_col and _rssi_col:
            for _c in [_lat_col, _lon_col, _rssi_col]:
                _df[_c] = pd.to_numeric(_df[_c], errors='coerce')
            _sel = _df[[_lat_col, _lon_col, _rssi_col]].dropna().reset_index(drop=True)

            # Take first NUM_RX_PLOT rows (same order as simulation)
            if NUM_RX_PLOT is not None:
                _sel = _sel.head(NUM_RX_PLOT)

            _rx_rows = list(zip(_sel[_lon_col], _sel[_lat_col], _sel[_rssi_col]))
            print(f'Loaded {len(_rx_rows)} RX points  '
                  f'(RSSI {float(_sel[_rssi_col].min()):.1f} – {float(_sel[_rssi_col].max()):.1f} dBm)')
        else:
            print(f'WARNING: could not find lat/lon/rssi columns. '
                  f'Available: {list(_df.columns)}')

# ── Plot ──────────────────────────────────────────────────────────────────
fig2d, ax2d = plt.subplots(figsize=(11, 10), dpi=FIG_DPI_2D)

# Building footprints
_gdf_plot = gdf_bld.copy()
if hasattr(_gdf_plot, 'crs') and _gdf_plot.crs and str(_gdf_plot.crs) != 'EPSG:4326':
    _gdf_plot = _gdf_plot.to_crs('EPSG:4326')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _gdf_plot.plot(ax=ax2d, facecolor='#d8d0c4', edgecolor='#999999',
                   linewidth=0.15, alpha=0.85, zorder=2)

ax2d.set_xlim(SCENE_WEST, SCENE_EAST)
ax2d.set_ylim(SCENE_SOUTH, SCENE_NORTH)
ax2d.set_aspect('equal')
ax2d.set_facecolor('#eef2f5')
ax2d.set_xlabel('Longitude', fontsize=10)
ax2d.set_ylabel('Latitude',  fontsize=10)
ax2d.tick_params(labelsize=8)
ax2d.grid(True, linestyle='--', linewidth=0.4, alpha=0.5, zorder=1)

# Scene bbox border
from matplotlib.patches import Rectangle as _Rect
ax2d.add_patch(_Rect((SCENE_WEST, SCENE_SOUTH),
                      SCENE_EAST - SCENE_WEST, SCENE_NORTH - SCENE_SOUTH,
                      linewidth=1.5, edgecolor='black', facecolor='none',
                      linestyle='--', zorder=6))

# RX dots coloured by RSSI
if _rx_rows:
    _lons_p = np.array([p[0] for p in _rx_rows])
    _lats_p = np.array([p[1] for p in _rx_rows])
    _rssi_p = np.array([p[2] for p in _rx_rows])
    _vmin_p = float(np.percentile(_rssi_p, 2))
    _vmax_p = float(np.percentile(_rssi_p, 98))
    _sc2d = ax2d.scatter(_lons_p, _lats_p, c=_rssi_p, s=2.5,
                         cmap='RdYlGn', vmin=_vmin_p, vmax=_vmax_p,
                         alpha=0.65, linewidths=0, zorder=5, label='RX (RSSI)')
    _cb2d = fig2d.colorbar(_sc2d, ax=ax2d, fraction=0.025, pad=0.01, shrink=0.7)
    _cb2d.set_label('RSSI (dBm)', fontsize=9)
    _cb2d.ax.tick_params(labelsize=8)

# TX star
if _tx_lon and _tx_lat:
    ax2d.plot(_tx_lon, _tx_lat, marker='*', markersize=16, color='red',
              markeredgecolor='darkred', markeredgewidth=0.8,
              zorder=10, label='TX (transmitter)')
    ax2d.annotate('TX', (_tx_lon, _tx_lat),
                  textcoords='offset points', xytext=(8, 5),
                  fontsize=9, color='darkred', fontweight='bold', zorder=11)

_n_bld = len(_gdf_plot) if '_gdf_plot' in dir() else 0
ax2d.set_title(
    f'Nottingham 915 MHz — OSM Scene Map\n'
    f'{_n_bld:,} buildings  |  {len(_rx_rows):,} RX (first {NUM_RX_PLOT})  |  '
    f'{(SCENE_EAST-SCENE_WEST)*111.32*np.cos(np.radians((SCENE_SOUTH+SCENE_NORTH)/2)):.1f} × '
    f'{(SCENE_NORTH-SCENE_SOUTH)*111.32:.1f} km',
    fontsize=11)

ax2d.legend(loc='upper right', fontsize=9, markerscale=3,
            framealpha=0.85, edgecolor='grey')

plt.tight_layout()

if SAVE_FIG_2D:
    _fig2d_path = os.path.join(BASE_DIR, 'scene_map_2d.png')
    fig2d.savefig(_fig2d_path, dpi=FIG_DPI_2D, bbox_inches='tight')
    print(f'Saved: {_fig2d_path}')

plt.show()
print('2D map done.')

## CELL 8 — Interactive 3D RSSI Heatmap (Plotly)


In [ ]:
# ============================================================
# CELL 8 — INTERACTIVE 3D: DEM TERRAIN + RSSI DOTS  (Plotly)
# ============================================================
# Layer order (bottom → top):
#   1. DEM terrain surface coloured by elevation  (terrain colorscale)
#   2. Building footprints extruded as grey 3D boxes
#   3. RX measurement dots coloured by RSSI floating at 1.5 m AGL
#   4. TX marker at antenna height
#
# Flat terrain (FLAT_TERRAIN=True): terrain surface is a flat grey plane.
# Real DEM (FLAT_TERRAIN=False): terrain undulates — run CELL 3 first.
#
# Requirements: pip install plotly scipy
# Run after CELL 3 (XX,YY,ZZ terrain grid) and CELL 7 (_rx_rows loaded).
# Saves: BASE_DIR/rssi_3d_dem.html  (open in any browser)
# ============================================================

try:
    import plotly.graph_objects as go
    _HAS_PLOTLY = True
except ImportError:
    _HAS_PLOTLY = False
    print('plotly not installed — run:  pip install plotly')

try:
    from scipy.interpolate import griddata as _griddata
    _HAS_SCIPY = True
except ImportError:
    _HAS_SCIPY = False
    print('scipy not installed — run:  pip install scipy')

if not _HAS_PLOTLY or not _HAS_SCIPY:
    raise SystemExit('Install plotly and scipy first.')

if not _rx_rows:
    raise SystemExit('No RX data — run CELL 7 first to load _rx_rows.')

if 'XX' not in dir():
    raise SystemExit('Terrain grid not found — run CELL 3 first.')

import numpy as np, warnings

# ── 1. Coordinate helpers ─────────────────────────────────────────────────
_cx = (SCENE_WEST  + SCENE_EAST)  / 2
_cy = (SCENE_SOUTH + SCENE_NORTH) / 2
_lon2m = 111320 * np.cos(np.radians(_cy))
_lat2m = 111320

def _to_m(lon, lat):
    return (lon - _cx) * _lon2m, (lat - _cy) * _lat2m

_x_span = (SCENE_EAST  - SCENE_WEST)  * _lon2m
_y_span = (SCENE_NORTH - SCENE_SOUTH) * _lat2m

# ── 2. DEM surface (subsample for performance) ────────────────────────────
_DEM_N = 200   # downsample terrain grid to 200×200 for Plotly rendering
N_full = XX.shape[0]
_step  = max(1, N_full // _DEM_N)
_XX_s  = XX[::_step, ::_step]
_YY_s  = YY[::_step, ::_step]
_ZZ_s  = ZZ[::_step, ::_step]

# Elevation colorscale — grey-green-brown terrain palette
_TERRAIN_CS = [
    [0.00, '#4a7c59'],   # low  → dark green
    [0.25, '#7aab6f'],   # green
    [0.50, '#c4b87a'],   # tan
    [0.75, '#a08060'],   # brown
    [1.00, '#d0c8b8'],   # high → light stone
]

print(f'DEM surface: {_XX_s.shape[0]}×{_XX_s.shape[1]} grid  '
      f'(Z range {float(_ZZ_s.min()):.1f}–{float(_ZZ_s.max()):.1f} m)')

# ── 3. RX scatter ─────────────────────────────────────────────────────────
_rx_x = np.array([(p[0] - _cx) * _lon2m for p in _rx_rows])
_rx_y = np.array([(p[1] - _cy) * _lat2m for p in _rx_rows])
_rx_r = np.array([p[2] for p in _rx_rows])

# Terrain height at each RX position (bilinear from subsampled grid)
_rx_z_terrain = _griddata(
    np.column_stack([_XX_s.ravel(), _YY_s.ravel()]), _ZZ_s.ravel(),
    np.column_stack([_rx_x, _rx_y]), method='linear', fill_value=0.0
)
_rx_z_plot = _rx_z_terrain + 1.5   # 1.5 m above terrain

_vmin = float(np.percentile(_rx_r, 2))
_vmax = float(np.percentile(_rx_r, 98))
print(f'RSSI range (2–98 pct): {_vmin:.1f} – {_vmax:.1f} dBm')

# ── 4. Building boxes ─────────────────────────────────────────────────────
print('Building 3D mesh ...')
_MAX_BLDS = 3000
_bld_count = 0

_gdf_wgs = gdf_bld.copy()
if hasattr(_gdf_wgs, 'crs') and _gdf_wgs.crs and str(_gdf_wgs.crs) != 'EPSG:4326':
    _gdf_wgs = _gdf_wgs.to_crs('EPSG:4326')

from shapely.geometry import Polygon as _Poly, MultiPolygon as _MPoly

_bx_all, _by_all, _bz_all = [], [], []
_bi_all, _bj_all, _bk_all = [], [], []
_v_off = 0

for _, _row in _gdf_wgs.iterrows():
    if _bld_count >= _MAX_BLDS:
        break
    _geom = _row.geometry
    if _geom is None or _geom.is_empty:
        continue
    if isinstance(_geom, _MPoly):
        _geom = max(_geom.geoms, key=lambda g: g.area)
    if not isinstance(_geom, _Poly):
        continue

    _h = DEFAULT_HEIGHT_M
    try:
        _hv = float(str(_row.get('height','0')).replace('m','').strip())
        if _hv > 1: _h = float(np.clip(_hv, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M))
    except: pass
    try:
        _lv = float(str(_row.get('building:levels','0')).strip())
        if _lv > 0: _h = float(np.clip(_lv * HEIGHT_PER_LEVEL_M, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M))
    except: pass

    _pts = list(_geom.exterior.coords)
    if len(_pts) < 4:
        continue
    if np.allclose(_pts[0], _pts[-1]):
        _pts = _pts[:-1]
    _n = len(_pts)

    # Base z from terrain at centroid
    _rcx = np.mean([p[0] for p in _pts])
    _rcy = np.mean([p[1] for p in _pts])
    _rmx, _rmy = _to_m(_rcx, _rcy)
    _base_z = float(_griddata(
        np.column_stack([_XX_s.ravel(), _YY_s.ravel()]), _ZZ_s.ravel(),
        [[_rmx, _rmy]], method='nearest'
    )[0])

    for _lon_v, _lat_v in _pts:
        _mx, _my = _to_m(_lon_v, _lat_v)
        _bx_all.append(_mx); _by_all.append(_my); _bz_all.append(_base_z)
    for _lon_v, _lat_v in _pts:
        _mx, _my = _to_m(_lon_v, _lat_v)
        _bx_all.append(_mx); _by_all.append(_my); _bz_all.append(_base_z + _h)

    for _i in range(_n):
        _j = (_i + 1) % _n
        _b0, _b1 = _v_off + _i,       _v_off + _j
        _t0, _t1 = _v_off + _n + _i,  _v_off + _n + _j
        _bi_all += [_b0, _b0]; _bj_all += [_b1, _t0]; _bk_all += [_t0, _t1]

    # Roof
    _c_idx = _v_off + 2*_n
    _bx_all.append(_rmx); _by_all.append(_rmy); _bz_all.append(_base_z + _h)
    for _i in range(_n):
        _j = (_i + 1) % _n
        _bi_all.append(_v_off + _n + _i)
        _bj_all.append(_v_off + _n + _j)
        _bk_all.append(_c_idx)

    _v_off += 2*_n + 1
    _bld_count += 1

print(f'  {_bld_count} buildings in mesh')

# ── 5. Assemble figure ────────────────────────────────────────────────────
_fig3d = go.Figure()

# DEM terrain surface
_fig3d.add_trace(go.Surface(
    x=_XX_s, y=_YY_s, z=_ZZ_s,
    colorscale=_TERRAIN_CS,
    showscale=True,
    colorbar=dict(
        title='Elevation (m)', thickness=12, len=0.45,
        x=1.02, y=0.75, yanchor='top',
    ),
    opacity=1.0,
    name='DEM Terrain',
    hovertemplate='Elev: %{z:.1f} m<extra>Terrain</extra>',
    lighting=dict(ambient=0.6, diffuse=0.8, specular=0.1, roughness=0.8),
    lightposition=dict(x=1000, y=2000, z=2000),
))

# Buildings
if _bx_all:
    _fig3d.add_trace(go.Mesh3d(
        x=_bx_all, y=_by_all, z=_bz_all,
        i=_bi_all, j=_bj_all, k=_bk_all,
        color='#b8b0a4', opacity=0.7,
        flatshading=True,
        lighting=dict(ambient=0.7, diffuse=0.6),
        name='Buildings',
        showscale=False,
        hoverinfo='skip',
    ))

# RX dots — RSSI colour
_fig3d.add_trace(go.Scatter3d(
    x=_rx_x, y=_rx_y, z=_rx_z_plot,
    mode='markers',
    marker=dict(
        size=3,
        color=_rx_r,
        colorscale='RdYlGn',
        cmin=_vmin, cmax=_vmax,
        opacity=0.85,
        showscale=True,
        colorbar=dict(
            title='RSSI (dBm)', thickness=12, len=0.45,
            x=1.02, y=0.25, yanchor='top',
        ),
        line=dict(width=0),
    ),
    name='RX (RSSI)',
    hovertemplate='RSSI: %{marker.color:.1f} dBm<extra>RX</extra>',
))

# TX
if _tx_lon and _tx_lat:
    _tx_mx, _tx_my = _to_m(_tx_lon, _tx_lat)
    _tx_z_base = float(_griddata(
        np.column_stack([_XX_s.ravel(), _YY_s.ravel()]), _ZZ_s.ravel(),
        [[_tx_mx, _tx_my]], method='nearest'
    )[0])
    _fig3d.add_trace(go.Scatter3d(
        x=[_tx_mx], y=[_tx_my], z=[_tx_z_base + 17.0],
        mode='markers+text',
        marker=dict(size=10, color='red', symbol='diamond',
                    line=dict(color='darkred', width=1)),
        text=['TX'], textposition='top center',
        textfont=dict(size=11, color='red'),
        name='TX',
    ))

# Layout
_z_exag = 5.0   # vertical exaggeration factor (makes terrain relief visible)
_fig3d.update_layout(
    title=dict(
        text='Nottingham 915 MHz — 3D Terrain + RSSI Coverage',
        font=dict(size=14)
    ),
    scene=dict(
        xaxis=dict(title='East–West (m)', showgrid=True, gridcolor='#cccccc'),
        yaxis=dict(title='North–South (m)', showgrid=True, gridcolor='#cccccc'),
        zaxis=dict(title='Height (m)', showgrid=True, gridcolor='#cccccc'),
        aspectmode='manual',
        aspectratio=dict(
            x=1.0,
            y=_y_span / _x_span,
            z=_z_exag * (float(_ZZ_s.max()) - float(_ZZ_s.min()) + 50) / _x_span
              if not FLAT_TERRAIN else 0.06
        ),
        camera=dict(
            eye=dict(x=0.0, y=-1.8, z=1.0),
            up=dict(x=0, y=0, z=1),
        ),
        bgcolor='#e8edf2',
    ),
    margin=dict(l=0, r=80, t=50, b=0),
    legend=dict(x=0.01, y=0.98, font=dict(size=10), bgcolor='rgba(255,255,255,0.7)'),
    width=1300, height=800,
    paper_bgcolor='white',
)

# ── 6. Save + show ────────────────────────────────────────────────────────
_html3d = os.path.join(BASE_DIR, 'rssi_3d_dem.html')
_fig3d.write_html(_html3d, include_plotlyjs='cdn')
print(f'Saved: {_html3d}')
print('Open in browser — rotate/zoom/hover to read elevation and RSSI.')
_fig3d.show()


## CELL 9 — Customised Antenna Patterns (Ofcom 915 MHz)

Defines TX and RX antenna pattern functions matching the Ofcom drive-test equipment,
then exports all link-budget parameters to `scene_parameters.json` for automatic
pick-up by the simulation notebook.


In [ ]:
# ============================================================
# CELL 9 — CUSTOMISED ANTENNA PATTERNS  (Ofcom 915 MHz)
# ============================================================
# Defines TX / RX antenna pattern callables matching the Ofcom
# drive-test equipment used in the nottingham915.csv dataset.
#
# TX : collinear omni mast — donut shaped, peak = TX_ANTENNA_GAIN_DBI
# RX : vehicle rooftop — effectively isotropic; equipment losses
#      (cable, splitter, BPF) folded into RX_EXTRA_GAIN_DB
#
# Pattern functions are TensorFlow callables accepted by Sionna's
# PlanarArray.  A safe fallback to 'hw_dipole' / 'iso' is provided
# if the Sionna version does not accept callables.
#
# After defining patterns, all link-budget parameters are written to
# scene_parameters.json so the simulation notebook reads them
# automatically.
# ============================================================

import numpy as np
import json as _json

# ── Half-wave dipole directivity (linear) ─────────────────────────────────
_D_HW = 1.6409   # D of ideal half-wave dipole (= 2.15 dBi in linear)

# ── TX : donut omni scaled to TX_ANTENNA_GAIN_DBI ─────────────────────────
def _tx_pattern_915(theta, phi):
    """
    Ofcom 915 MHz TX — collinear omni mast.
    Pattern: donut (half-wave dipole shape) scaled so peak = TX_ANTENNA_GAIN_DBI.
    F_theta = scale * cos(pi/2 * cos(theta)) / sin(theta)
    F_phi   = 0
    """
    import tensorflow as tf
    _scale = np.float32((10 ** (TX_ANTENNA_GAIN_DBI / 10) / _D_HW) ** 0.5)
    cos_t  = tf.cos(theta)
    sin_t  = tf.sin(theta)
    safe_s = tf.where(tf.abs(sin_t) < 1e-6,
                      tf.ones_like(sin_t) * 1e-6, sin_t)
    f_theta = tf.cast(_scale * tf.cos(np.float32(np.pi / 2) * cos_t) / safe_s,
                      tf.complex64)
    f_phi   = tf.zeros_like(f_theta)
    return f_theta, f_phi

# ── RX : isotropic — equipment losses in RX_EXTRA_GAIN_DB ─────────────────
def _rx_pattern_915(theta, phi):
    """
    Ofcom 915 MHz RX — vehicle rooftop omni.
    Treated as isotropic (0 dBi); drive-test NLOS paths arrive from varied
    elevations so a directional pattern would introduce azimuth bias.
    Cable / splitter / filter losses are captured in RX_EXTRA_GAIN_DB.
    """
    import tensorflow as tf
    _iso = tf.cast(tf.ones_like(theta) / np.float32(np.sqrt(4.0 * np.pi)),
                   tf.complex64)
    f_phi = tf.zeros_like(_iso)
    return _iso, f_phi

# ── Helper: build PlanarArray with fallback ───────────────────────────────
def _make_antenna_array(pattern_fn, fallback='hw_dipole'):
    """
    Try to create a 1×1 PlanarArray with a callable pattern.
    Falls back to a named built-in if Sionna rejects the callable.
    Returns (array, mode_string).
    """
    try:
        from sionna.rt import PlanarArray
        arr = PlanarArray(num_rows=1, num_cols=1,
                          vertical_spacing=0.5, horizontal_spacing=0.5,
                          pattern=pattern_fn, polarization='V')
        return arr, 'callable'
    except Exception as _e:
        try:
            from sionna.rt import PlanarArray
            arr = PlanarArray(num_rows=1, num_cols=1,
                              vertical_spacing=0.5, horizontal_spacing=0.5,
                              pattern=fallback, polarization='V')
            return arr, f'fallback={fallback} ({_e})'
        except Exception as _e2:
            return None, f'failed ({_e2})'

# ── Verify pattern shape at sample angles ─────────────────────────────────
print('Antenna pattern check (915 MHz):')
print(f'  TX pattern : donut omni  peak = {TX_ANTENNA_GAIN_DBI:+.1f} dBi')
print(f'  RX pattern : isotropic   system gain = {RX_EXTRA_GAIN_DB:.1f} dB')
print(f'  Frequency  : {FREQUENCY_HZ/1e6:.2f} MHz')
print(f'  TX AGL     : {TX_AGL_M:.1f} m   RX AGL : {RX_AGL_M:.1f} m')
print(f'  TX EIRP    : {TX_CONDUCTED_DBM + TX_ANTENNA_GAIN_DBI:.1f} dBm  '
      f'(conducted={TX_CONDUCTED_DBM:.1f} + antenna={TX_ANTENNA_GAIN_DBI:.1f})')
print()

# Evaluate TX pattern at horizon (theta=pi/2) and zenith (theta=0)
try:
    import tensorflow as tf
    _th_h = tf.constant([np.pi/2], dtype=tf.float32)
    _ph_h = tf.constant([0.0],     dtype=tf.float32)
    _fth, _ = _tx_pattern_915(_th_h, _ph_h)
    _g_horizon = 10 * np.log10(float(tf.reduce_sum(tf.abs(_fth)**2).numpy()) * _D_HW)
    print(f'  TX gain at horizon (theta=90°): {_g_horizon:.2f} dBi  '
          f'(expected {TX_ANTENNA_GAIN_DBI:.2f} dBi)')

    _th_z = tf.constant([0.01], dtype=tf.float32)   # avoid exact 0
    _fth_z, _ = _tx_pattern_915(_th_z, _ph_h)
    _g_zenith = 10 * np.log10(max(float(tf.reduce_sum(tf.abs(_fth_z)**2).numpy()), 1e-12) * _D_HW)
    print(f'  TX gain at zenith  (theta≈0°) : {_g_zenith:.2f} dBi  (expected deep null)')
except Exception as _e:
    print(f'  Pattern evaluation skipped (TensorFlow not available in this env): {_e}')

# ── Optional: build arrays if sionna.rt is importable ─────────────────────
print()
try:
    _tx_arr, _tx_mode = _make_antenna_array(_tx_pattern_915, fallback='hw_dipole')
    _rx_arr, _rx_mode = _make_antenna_array(_rx_pattern_915, fallback='iso')
    print(f'  PlanarArray TX : [{_tx_mode}]')
    print(f'  PlanarArray RX : [{_rx_mode}]')
    print('  (Arrays ready — assign to scene.tx_array / scene.rx_array in sim notebook)')
    # Store for downstream cells
    _tx_array_915 = _tx_arr
    _rx_array_915 = _rx_arr
except Exception as _e:
    print(f'  sionna.rt not available in scene builder env — arrays not built: {_e}')
    print('  Pattern functions _tx_pattern_915 / _rx_pattern_915 are defined and')
    print('  can be copied to / imported by the simulation notebook.')

# ── Export antenna + link budget params to scene_parameters.json ──────────
_params_path = os.path.join(BASE_DIR, 'scene_parameters.json')
try:
    with open(_params_path) as _f:
        _meta = _json.load(_f)
except FileNotFoundError:
    _meta = {}

_meta.update({
    'frequency_hz'         : FREQUENCY_HZ,
    'antenna_pattern'      : ANTENNA_PATTERN,
    'tx_agl_m'             : TX_AGL_M,
    'rx_agl_m'             : RX_AGL_M,
    'tx_conducted_dbm'     : TX_CONDUCTED_DBM,
    'tx_antenna_gain_dbi'  : TX_ANTENNA_GAIN_DBI,
    'tx_eirp_dbm'          : TX_CONDUCTED_DBM + TX_ANTENNA_GAIN_DBI,
    'rx_extra_gain_db'     : RX_EXTRA_GAIN_DB,
    'site_correction_db'   : SITE_CORRECTION_DB,
})

with open(_params_path, 'w') as _f:
    _json.dump(_meta, _f, indent=2)

print()
print(f'scene_parameters.json updated: {_params_path}')
print(json.dumps({k: v for k, v in _meta.items()
                  if k in ('frequency_hz','antenna_pattern','tx_eirp_dbm',
                           'tx_agl_m','rx_agl_m','rx_extra_gain_db')}, indent=2))

---

# Section B — Blender Scene Conversion

These cells are **standalone** and can be run independently of Section A. They convert Blender-exported XML or OSM-built scenes into the target Sionna format.

Set the input/output paths at the top of each cell before running.


## CELL B1 — Blender XML → Sionna 0.19

Reads a Blender-exported or Sionna 2.0 scene XML and writes a Sionna 0.19 compatible version:
- Strips `mat-` prefix from ITU material IDs
- Replaces `diffuse`/`twosided` BSDFs with `conductor` + ITU-R P.2040-2 `eta`/`k`
- Maps colour-named materials to the nearest ITU equivalent

**Inputs to set:** `BLENDER_XML_IN`, `SIONNA19_XML_OUT`


In [ ]:
# ============================================================
# CELL B1 — CONVERT BLENDER / SIONNA 2.0 XML → SIONNA 0.19
# ============================================================

BLENDER_XML_IN  = '/path/to/your/blender_scene.xml'   # ← set this
SIONNA19_XML_OUT = os.path.join(SCENE_DIR, 'scene_from_blender.xml')

# ── ITU-R P.2040-2 EM parameters (relative_permittivity, conductivity_S/m) ──
ITU_EM = {
    'itu_concrete'  : (5.31,  0.092),
    'itu_brick'     : (3.75,  0.038),
    'itu_glass'     : (6.27,  0.000),
    'itu_wood'      : (1.99,  0.000),
    'itu_metal'     : (1.00,  1.0e7),
    'itu_asphalt'   : (2.56,  0.000),
    'itu_wet_ground': (30.0,  0.020),
    'itu_water'     : (80.0,  0.020),  # mapped → itu_wet_ground below
}

# ── Material name mapping: Blender/Sionna2 ID → Sionna 0.19 ITU name ────────
# Add custom entries here as needed.
MAT_MAP = {
    # Strip mat- from known ITU names
    'mat-itu_brick'     : 'itu_brick',
    'mat-itu_concrete'  : 'itu_concrete',
    'mat-itu_glass'     : 'itu_glass',
    'mat-itu_wood'      : 'itu_wood',
    'mat-itu_metal'     : 'itu_metal',
    'mat-itu_asphalt'   : 'itu_asphalt',
    'mat-itu_wet_ground': 'itu_wet_ground',
    'mat-itu_water'     : 'itu_wet_ground',
    'mat-itu_very_dry_ground'   : 'itu_wet_ground',
    'mat-itu_medium_dry_ground' : 'itu_wet_ground',
    # Colour / custom → nearest ITU
    'mat-white'       : 'itu_concrete',
    'mat-grey'        : 'itu_concrete',
    'mat-darkgrey'    : 'itu_concrete',
    'mat-red'         : 'itu_brick',
    'mat-salmon'      : 'itu_brick',
    'mat-tan'         : 'itu_brick',
    'mat-brown'       : 'itu_brick',
    'mat-d5b9a3'      : 'itu_brick',
    'mat-85552e'      : 'itu_brick',
    'mat-ff9e6b'      : 'itu_brick',
    'mat-green'       : 'itu_concrete',
    'mat-lime'        : 'itu_concrete',
    'mat-forest'      : 'itu_concrete',
    'mat-vegetation'  : 'itu_concrete',
    'mat-areas_pedestrian' : 'itu_asphalt',
    'mat-areas_service'    : 'itu_asphalt',
    'mat-areas_railways'   : 'itu_asphalt',
    'mat-areas_footway'    : 'itu_asphalt',
    'mat-areas_road'       : 'itu_asphalt',
    'mat-road'             : 'itu_asphalt',
    'mat-pavement'         : 'itu_asphalt',
}

def resolve_mat(mat_id):
    """Return Sionna 0.19 ITU name for a Blender material ID."""
    if mat_id in MAT_MAP:
        return MAT_MAP[mat_id]
    # Auto-strip mat- prefix if it looks like an ITU name
    stripped = mat_id.replace('mat-', '')
    if stripped in ITU_EM:
        return stripped
    # Fallback: unknown material → itu_concrete
    print(f'  [WARN] unknown material "{mat_id}" → itu_concrete (add to MAT_MAP to override)')
    return 'itu_concrete'

# ── Parse input XML ───────────────────────────────────────────────────────────
import xml.etree.ElementTree as ET
assert os.path.exists(BLENDER_XML_IN), f'Input not found: {BLENDER_XML_IN}'

tree = ET.parse(BLENDER_XML_IN)
root = tree.getroot()

# Collect all material IDs used in shapes (only write materials actually used)
used_mat_ids = set()
for shape in root.findall('.//shape'):
    ref = shape.find('ref[@name="bsdf"]')
    if ref is not None:
        used_mat_ids.add(ref.get('id'))

print(f'Input   : {BLENDER_XML_IN}')
print(f'Shapes  : {len(root.findall(".//shape"))}')
print(f'Mat IDs : {sorted(used_mat_ids)}')
print()

# Resolve to Sionna 0.19 names
mat_resolved = {mid: resolve_mat(mid) for mid in used_mat_ids}
used_itu = sorted(set(mat_resolved.values()))
print(f'Resolved ITU materials: {used_itu}')
print()
for orig, itu in sorted(mat_resolved.items()):
    if orig != itu:
        print(f'  {orig:<35} → {itu}')

# ── Build output XML lines ────────────────────────────────────────────────────
out = []
out.append('<?xml version="1.0" encoding="utf-8"?>')
out.append('<scene version="2.1.0">')
out.append('')
out.append('  <!-- ── ITU-R P.2040-2 Materials (Sionna 0.19) ────────── -->')

for itu_name in used_itu:
    eps, sigma = ITU_EM.get(itu_name, (5.31, 0.092))
    out.append(f'  <bsdf type="conductor" id="{itu_name}">')
    out.append(f'    <float name="eta" value="{eps}"/>')
    out.append(f'    <float name="k"   value="{sigma}"/>')
    out.append(f'  </bsdf>')
    out.append('')

out.append('  <!-- ── Shapes ─────────────────────────────────────────── -->')

for shape in root.findall('.//shape'):
    stype    = shape.get('type', 'ply')
    sid      = shape.get('id', '')
    filename = shape.findtext('string[@name="filename"]') or ''
    ref      = shape.find('ref[@name="bsdf"]')
    if ref is None:
        continue
    orig_mat  = ref.get('id')
    itu_mat   = mat_resolved.get(orig_mat, 'itu_concrete')

    out.append(f'  <shape type="{stype}" id="{sid}">')
    out.append(f'    <string name="filename" value="{filename}"/>')
    out.append(f'    <ref id="{itu_mat}" name="bsdf"/>')
    out.append(f'    <boolean name="face_normals" value="true"/>')
    out.append(f'  </shape>')

out.append('')
out.append('</scene>')

with open(SIONNA19_XML_OUT, 'w') as f:
    f.write('\n'.join(out))

print(f'\nWrote: {SIONNA19_XML_OUT}')
print(f'  Shapes    : {len(root.findall(".//shape"))}')
print(f'  Materials : {len(used_itu)} ITU materials')


## CELL B2 — Blender XML → Sionna 2.0

Copies the original Blender XML and makes the minimum changes needed for Sionna 2.0:
- Maps colour/custom material IDs to nearest `mat-itu_*` equivalent
- Adds missing `mat-itu_*` BSDF definitions (twosided/diffuse)
- Does **not** modify the original Blender file

**Inputs to set:** `BLENDER_XML_IN`, `SIONNA2_XML_OUT`


In [ ]:
# ============================================================
# CELL B2 — BLENDER XML → SIONNA 2.0 COMPATIBLE  (copy original)
# ============================================================
# Reads the Blender-exported XML, maps non-ITU colour materials
# to the nearest mat-itu_* name, writes a new file.
# Original XML is never modified.
# ============================================================
import shutil, xml.etree.ElementTree as ET
from lxml import etree as _lxml   # preserves formatting; falls back to stdlib below

BLENDER_XML_IN   = '/path/to/your/blender_scene.xml'   # ← set this
SIONNA2_XML_OUT  = os.path.join(SCENE_DIR, 'scene_sionna2_from_blender.xml')

# ── Material colours for any mat-itu_* BSDF we may need to add ───────────────
ITU_COLOURS_S2 = {
    'mat-itu_concrete'  : '0.539 0.539 0.539',
    'mat-itu_brick'     : '1.000 0.498 0.055',
    'mat-itu_glass'     : '0.596 0.875 0.541',
    'mat-itu_wood'      : '0.043 0.580 0.184',
    'mat-itu_metal'     : '0.220 0.220 0.254',
    'mat-itu_asphalt'   : '0.200 0.200 0.200',
    'mat-itu_wet_ground': '0.910 0.569 0.055',
}

# ── Colour / custom material → nearest mat-itu_* ─────────────────────────────
MAT_MAP_S2 = {
    'mat-white'       : 'mat-itu_concrete',
    'mat-grey'        : 'mat-itu_concrete',
    'mat-darkgrey'    : 'mat-itu_concrete',
    'mat-red'         : 'mat-itu_brick',
    'mat-salmon'      : 'mat-itu_brick',
    'mat-tan'         : 'mat-itu_brick',
    'mat-brown'       : 'mat-itu_brick',
    'mat-d5b9a3'      : 'mat-itu_brick',
    'mat-85552e'      : 'mat-itu_brick',
    'mat-ff9e6b'      : 'mat-itu_brick',
    'mat-green'       : 'mat-itu_concrete',
    'mat-lime'        : 'mat-itu_concrete',
    'mat-forest'      : 'mat-itu_concrete',
    'mat-vegetation'  : 'mat-itu_concrete',
    'mat-itu_water'   : 'mat-itu_wet_ground',
    'mat-areas_pedestrian' : 'mat-itu_asphalt',
    'mat-areas_service'    : 'mat-itu_asphalt',
    'mat-areas_railways'   : 'mat-itu_asphalt',
    'mat-areas_footway'    : 'mat-itu_asphalt',
    'mat-areas_road'       : 'mat-itu_asphalt',
    'mat-road'             : 'mat-itu_asphalt',
    'mat-pavement'         : 'mat-itu_asphalt',
}

def resolve_s2(mat_id):
    if mat_id in MAT_MAP_S2:
        return MAT_MAP_S2[mat_id]
    if mat_id.startswith('mat-itu_'):
        return mat_id   # already a valid Sionna 2.0 ITU name
    print(f'  [WARN] unknown material "{mat_id}" → mat-itu_concrete (add to MAT_MAP_S2 to override)')
    return 'mat-itu_concrete'

# ── Step 1: copy original ────────────────────────────────────────────────────
assert os.path.exists(BLENDER_XML_IN), f'Input not found: {BLENDER_XML_IN}'
shutil.copy2(BLENDER_XML_IN, SIONNA2_XML_OUT)
print(f'Copied  : {BLENDER_XML_IN}')
print(f'      → : {SIONNA2_XML_OUT}')

# ── Step 2: parse the COPY (never touch original) ────────────────────────────
try:
    parser = _lxml.XMLParser(remove_blank_text=False)
    tree   = _lxml.parse(SIONNA2_XML_OUT, parser)
    root   = tree.getroot()
    USE_LXML = True
except Exception:
    tree   = ET.parse(SIONNA2_XML_OUT)
    root   = tree.getroot()
    USE_LXML = False

# ── Step 3: collect all shape refs and resolve mapping ───────────────────────
shapes = root.findall('.//shape')
used_orig = set()
for s in shapes:
    ref = s.find('ref[@name="bsdf"]')
    if ref is not None:
        used_orig.add(ref.get('id'))

mat_map_applied = {m: resolve_s2(m) for m in used_orig}
remapped = {k: v for k, v in mat_map_applied.items() if k != v}
new_itu  = set(mat_map_applied.values()) - used_orig  # ITU BSDFs we need to add

print(f'\nShapes          : {len(shapes)}')
print(f'Original mat IDs: {sorted(used_orig)}')
if remapped:
    print(f'\nRemapped materials:')
    for orig, new in sorted(remapped.items()):
        print(f'  {orig:<35} → {new}')
if new_itu:
    print(f'\nNew BSDF definitions to add: {sorted(new_itu)}')

# ── Step 4: add missing mat-itu_* BSDF definitions ───────────────────────────
existing_bsdf_ids = {b.get('id') for b in root.findall('.//bsdf')}
for itu_id in sorted(new_itu):
    if itu_id in existing_bsdf_ids:
        continue
    colour = ITU_COLOURS_S2.get(itu_id, '0.5 0.5 0.5')
    if USE_LXML:
        bsdf_el  = _lxml.SubElement(root, 'bsdf', type='twosided', id=itu_id, name=itu_id)
        inner    = _lxml.SubElement(bsdf_el, 'bsdf', type='diffuse')
        rgb_el   = _lxml.SubElement(inner, 'rgb', value=colour, name='reflectance')
    else:
        bsdf_el = ET.SubElement(root, 'bsdf', {'type': 'twosided', 'id': itu_id, 'name': itu_id})
        inner   = ET.SubElement(bsdf_el, 'bsdf', {'type': 'diffuse'})
        ET.SubElement(inner, 'rgb', {'value': colour, 'name': 'reflectance'})
    print(f'  Added BSDF: {itu_id}')

# ── Step 5: update shape refs + ensure face_normals ──────────────────────────
for shape in shapes:
    ref = shape.find('ref[@name="bsdf"]')
    if ref is None:
        continue
    orig_id = ref.get('id')
    new_id  = mat_map_applied.get(orig_id, orig_id)
    if new_id != orig_id:
        ref.set('id', new_id)
    # Add face_normals if missing
    fn = shape.find('boolean[@name="face_normals"]')
    if fn is None:
        if USE_LXML:
            _lxml.SubElement(shape, 'boolean', name='face_normals', value='true')
        else:
            ET.SubElement(shape, 'boolean', {'name': 'face_normals', 'value': 'true'})

# ── Step 6: write output ─────────────────────────────────────────────────────
if USE_LXML:
    tree.write(SIONNA2_XML_OUT, pretty_print=True, xml_declaration=True, encoding='utf-8')
else:
    ET.indent(tree, space='  ')
    tree.write(SIONNA2_XML_OUT, xml_declaration=True, encoding='unicode')


# ── Step 7: merge PLYs per material (lossless — same geometry, fewer files) ──
MERGE_PLYS = True   # set False to skip merging and keep original per-building PLYs

if MERGE_PLYS:
    import numpy as np
    _input_dir   = os.path.dirname(os.path.abspath(BLENDER_XML_IN))
    _scene_dir   = os.path.dirname(os.path.abspath(SIONNA2_XML_OUT))
    _meshes_dir  = os.path.join(_input_dir, 'meshes')
    _merged_dir  = os.path.join(_scene_dir, 'meshes_merged')
    print(f'Input dir   : {_input_dir}')
    print(f'Meshes dir  : {_meshes_dir}  exists={os.path.exists(_meshes_dir)}')
    os.makedirs(_merged_dir, exist_ok=True)

    def _read_ply_np(path):
        """Read ASCII or binary PLY → (verts, faces) numpy arrays."""
        import struct
        with open(path, 'rb') as f:
            lines = []
            while True:
                line = f.readline().decode('ascii', errors='ignore').strip()
                lines.append(line)
                if line == 'end_header':
                    break
            n_verts = next(int(l.split()[-1]) for l in lines if l.startswith('element vertex'))
            n_faces = next(int(l.split()[-1]) for l in lines if l.startswith('element face'))
            is_bin  = any('binary' in l for l in lines)
            if is_bin:
                raw  = f.read()
                off  = 0
                verts = np.frombuffer(raw[off:off+n_verts*12], dtype=np.float32).reshape(-1,3)
                off += n_verts * 12
                faces = []
                for _ in range(n_faces):
                    n  = struct.unpack_from('B', raw, off)[0]; off += 1
                    fs = struct.unpack_from(f'{n}i', raw, off); off += 4*n
                    if n == 3: faces.append(fs)
                    elif n == 4:
                        faces += [(fs[0],fs[1],fs[2]),(fs[0],fs[2],fs[3])]
                faces = np.array(faces, dtype=np.int32)
            else:
                verts, faces = [], []
                for _ in range(n_verts):
                    verts.append(list(map(float, f.readline().split()[:3])))
                for _ in range(n_faces):
                    row = list(map(int, f.readline().split()))
                    if row[0] == 3: faces.append(row[1:4])
                    elif row[0] == 4:
                        faces += [row[1:4], [row[1],row[3],row[4]]]
                verts = np.array(verts, np.float32)
                faces = np.array(faces, np.int32)
        return verts, faces

    def _write_ply_bin(path, verts, faces):
        with open(path, 'wb') as f:
            hdr = (f"ply\nformat binary_little_endian 1.0\n"
                   f"element vertex {len(verts)}\n"
                   f"property float x\nproperty float y\nproperty float z\n"
                   f"element face {len(faces)}\n"
                   f"property list uchar int vertex_indices\nend_header\n")
            f.write(hdr.encode())
            f.write(verts.astype(np.float32).tobytes())
            for fc in faces:
                f.write(bytes([3]) + np.array(fc, np.int32).tobytes())

    # Group shapes by resolved material
    if USE_LXML:
        _root2 = tree.getroot()
        _shapes2 = _root2.findall('.//shape')
    else:
        _shapes2 = tree.getroot().findall('.//shape')

    mat_groups = {}   # mat_id -> list of ply paths
    for s in _shapes2:
        ref = s.find('ref[@name="bsdf"]')
        fn  = s.findtext('string[@name="filename"]') or ''
        if ref is None or not fn: continue
        mat = ref.get('id')
        # Try multiple locations
        _candidates = [
            os.path.join(_input_dir, fn),
            os.path.join(_input_dir, os.path.basename(fn)),
            os.path.join(_meshes_dir, os.path.basename(fn)),
            os.path.join(_scene_dir, fn),
        ]
        ply_path = next((p for p in _candidates if os.path.exists(p)), None)
        if ply_path:
            mat_groups.setdefault(mat, []).append(ply_path)

    # Debug: show first 3 shapes
    for _ds in _shapes2[:3]:
        _dfn  = _ds.findtext('string[@name="filename"]') or 'NO FILENAME'
        _dref = _ds.find('ref[@name="bsdf"]')
        _dmat = _dref.get('id') if _dref is not None else 'NO REF'
        _dp   = os.path.join(_input_dir, _dfn)
        print(f'  [debug] fn={_dfn}  mat={_dmat}  exists={os.path.exists(_dp)}')
    print(f'\nMerging {len(_shapes2)} shapes into {len(mat_groups)} PLYs ...')
    merged_plys = {}   # mat_id -> relative path in merged_dir

    for mat, paths in mat_groups.items():
        all_v, all_f, offset = [], [], 0
        for p in paths:
            try:
                v, f = _read_ply_np(p)
                all_f.append(f + offset)
                all_v.append(v)
                offset += len(v)
            except Exception as _e:
                print(f'  [skip] {os.path.basename(p)}: {_e}')
        if not all_v: continue
        V = np.concatenate(all_v, axis=0).astype(np.float32)
        F = np.concatenate(all_f, axis=0).astype(np.int32)
        out_name = f'{mat}.ply'
        out_path = os.path.join(_merged_dir, out_name)
        _write_ply_bin(out_path, V, F)
        merged_plys[mat] = f'meshes_merged/{out_name}'
        print(f'  {mat}: {len(paths)} PLYs → {len(V)} verts, {len(F)} faces → {out_name}')

    # Rewrite XML with merged shapes
    if USE_LXML:
        from lxml import etree as _lxml2
        _root3 = tree.getroot()
        # Remove all shape elements
        for s in _root3.findall('.//shape'):
            _root3.remove(s)
        # Add one shape per merged material
        for mat, rel_path in merged_plys.items():
            colour = ITU_COLOURS_S2.get(mat, '0.5 0.5 0.5')
            sh = _lxml2.SubElement(_root3, 'shape', type='ply', id=f'mesh-{mat}')
            _lxml2.SubElement(sh, 'string', name='filename', value=rel_path)
            _lxml2.SubElement(sh, 'ref', id=mat, name='bsdf')
            _lxml2.SubElement(sh, 'boolean', name='face_normals', value='true')
        tree.write(SIONNA2_XML_OUT, pretty_print=True, xml_declaration=True, encoding='utf-8')
    else:
        _root3 = tree.getroot()
        for s in list(_root3.findall('shape')):
            _root3.remove(s)
        for mat, rel_path in merged_plys.items():
            sh = ET.SubElement(_root3, 'shape', {'type':'ply','id':f'mesh-{mat}'})
            ET.SubElement(sh, 'string', {'name':'filename','value':rel_path})
            ET.SubElement(sh, 'ref', {'id':mat,'name':'bsdf'})
            ET.SubElement(sh, 'boolean', {'name':'face_normals','value':'true'})
        ET.indent(tree, space='  ')
        tree.write(SIONNA2_XML_OUT, xml_declaration=True, encoding='unicode')

    print(f'\nMerge complete: {len(_shapes2)} shapes → {len(merged_plys)} merged PLYs')
    print(f'  Merged PLYs : {_merged_dir}')
    print(f'  Scene XML   : {SIONNA2_XML_OUT}  (updated)')

print(f'\nWrote : {SIONNA2_XML_OUT}')
print(f'  Shapes    : {len(shapes)}')
print(f'  Materials : {len(set(mat_map_applied.values()))} unique ITU materials')
print(f'  Original  : {BLENDER_XML_IN}  ← untouched')


## CELL B3 — OSM Scene → Sionna 2.0 Export

Generates a Sionna 2.0 compatible scene XML from the OSM-built scene (produced by Section A). Uses `mat-` prefixed material IDs, `twosided/diffuse` BSDFs for rendering, and `face_normals=true` on all shapes.

**Requires:** Section A cells 0–5 to have been run first (SCENE_DIR, MESH_DIR must be set).


In [ ]:
# ============================================================
# CELL B3 — WRITE SCENE_SIONNA2.XML  (Sionna 2.0 / Mitsuba 2.1.0)
# ============================================================
# Generates a Sionna 2.0 compatible scene XML alongside the
# existing Sionna 0.19 scene.xml.
# Key differences vs 0.19:
#   - Material IDs prefixed with "mat-"  (mat-itu_brick etc.)
#   - BSDF type = twosided/diffuse + rgb  (visual colour for rendering)
#   - Shapes have id + face_normals=true
#   - Integrator, emitter, sensor blocks included
# ============================================================

# ── Reconstruct mat_plys and config if running standalone ────────────────────
import os
if 'TERRAIN_MATERIAL' not in dir():
    TERRAIN_MATERIAL = 'itu_wet_ground'
if 'mat_plys' not in dir():
    mat_plys = {}
    for _fname in os.listdir(MESH_DIR):
        if _fname.startswith('bld_') and _fname.endswith('.ply'):
            _mat = _fname[4:-4]
            mat_plys[_mat] = [('meshes/' + _fname, 'buildings')]
    print(f'Reconstructed mat_plys from disk: {list(mat_plys.keys())}')
if 'center_lat' not in dir():
    import json as _json
    _p = os.path.join(BASE_DIR, 'scene_parameters.json')
    if os.path.exists(_p):
        _meta = _json.load(open(_p))
        center_lat = _meta['scene_center_lat']
        center_lon = _meta['scene_center_lon']
        UTM_EPSG   = _meta.get('utm_epsg', 32630)
    else:
        center_lat = center_lon = 0.0; UTM_EPSG = 32630

# ── ITU visual colours (for Mitsuba rendering — not EM properties) ──────────
ITU_COLOURS = {
    'itu_concrete'  : '0.539 0.539 0.539',
    'itu_brick'     : '1.000 0.498 0.055',
    'itu_glass'     : '0.596 0.875 0.541',
    'itu_wood'      : '0.043 0.580 0.184',
    'itu_metal'     : '0.220 0.220 0.254',
    'itu_asphalt'   : '0.200 0.200 0.200',
    'itu_wet_ground': '0.910 0.569 0.055',
    'itu_very_dry_ground' : '0.498 0.498 0.498',
    'itu_medium_dry_ground': '0.780 0.780 0.780',
}

used_mats2 = set(mat_plys.keys()) | {TERRAIN_MATERIAL}

lines2 = []
lines2.append('<?xml version="1.0" ?>')
lines2.append('<scene version="2.1.0">')
lines2.append('')

# ── Metadata defaults ────────────────────────────────────────────────────────
lines2.append('  <!-- ── Scene metadata ──────────────────────────────────── -->')
lines2.append(f'  <default name="scenegen_version"     value="1.0.0"/>')
lines2.append(f'  <default name="scenegen_min_lat"     value="{SCENE_SOUTH}"/>')
lines2.append(f'  <default name="scenegen_max_lat"     value="{SCENE_NORTH}"/>')
lines2.append(f'  <default name="scenegen_min_lon"     value="{SCENE_WEST}"/>')
lines2.append(f'  <default name="scenegen_max_lon"     value="{SCENE_EAST}"/>')
lines2.append(f'  <default name="scenegen_center_lat"  value="{center_lat}"/>')
lines2.append(f'  <default name="scenegen_center_lon"  value="{center_lon}"/>')
lines2.append(f'  <default name="scenegen_UTM_zone"    value="EPSG:{UTM_EPSG}"/>')
lines2.append(f'  <default name="scenegen_ground_material"  value="mat-{TERRAIN_MATERIAL}"/>')
lines2.append('')

# ── Integrator ───────────────────────────────────────────────────────────────
lines2.append('  <!-- ── Integrator ──────────────────────────────────────── -->')
lines2.append('  <integrator type="path">')
lines2.append('    <integer name="max_depth" value="12"/>')
lines2.append('  </integrator>')
lines2.append('')

# ── Materials — twosided diffuse (visual) ────────────────────────────────────
lines2.append('  <!-- ── ITU-R Materials (visual BSDF for rendering) ─────── -->')
for mat_name in sorted(used_mats2):
    colour = ITU_COLOURS.get(mat_name, '0.5 0.5 0.5')
    lines2.append(f'  <bsdf type="twosided" id="mat-{mat_name}">')
    lines2.append(f'    <bsdf type="diffuse">')
    lines2.append(f'      <rgb value="{colour}" name="reflectance"/>')
    lines2.append(f'    </bsdf>')
    lines2.append(f'  </bsdf>')
    lines2.append('')

# ── Emitter ──────────────────────────────────────────────────────────────────
lines2.append('  <!-- ── Environment ─────────────────────────────────────── -->')
lines2.append('  <emitter type="constant" id="World">')
lines2.append('    <rgb value="1.0 1.0 1.0" name="radiance"/>')
lines2.append('  </emitter>')
lines2.append('')

# ── Camera ───────────────────────────────────────────────────────────────────
lines2.append('  <!-- ── Camera (top-down overview) ─────────────────────── -->')
lines2.append('  <sensor type="perspective" id="Camera">')
lines2.append('    <string name="fov_axis" value="x"/>')
lines2.append('    <float  name="fov"      value="42.855"/>')
lines2.append('    <float  name="near_clip" value="0.1"/>')
lines2.append('    <float  name="far_clip"  value="10000.0"/>')
lines2.append('    <transform name="to_world">')
lines2.append('      <rotate z="1" angle="-90"/>')
lines2.append('      <translate value="0 0 500"/>')
lines2.append('    </transform>')
lines2.append('    <sampler type="independent">')
lines2.append('      <integer name="sample_count" value="4096"/>')
lines2.append('    </sampler>')
lines2.append('    <film type="hdrfilm">')
lines2.append('      <integer name="width"  value="1024"/>')
lines2.append('      <integer name="height" value="1024"/>')
lines2.append('    </film>')
lines2.append('  </sensor>')
lines2.append('')

# ── Terrain shape ─────────────────────────────────────────────────────────────
lines2.append('  <!-- ── Terrain ─────────────────────────────────────────── -->')
lines2.append('  <shape type="ply" id="mesh-ground">')
lines2.append('    <string name="filename" value="meshes/terrain.ply"/>')
lines2.append(f'    <ref id="mat-{TERRAIN_MATERIAL}" name="bsdf"/>')
lines2.append('    <boolean name="face_normals" value="true"/>')
lines2.append('  </shape>')
lines2.append('')

# ── Building shapes ───────────────────────────────────────────────────────────
lines2.append('  <!-- ── Buildings (one merged PLY per material) ──────────── -->')
for mat_name, ply_list in sorted(mat_plys.items()):
    for ply_path, role in ply_list:
        mesh_id = 'mesh-' + ply_path.split('/')[-1].replace('.ply', '')
        lines2.append(f'  <shape type="ply" id="{mesh_id}">')
        lines2.append(f'    <string name="filename" value="{ply_path}"/>')
        lines2.append(f'    <ref id="mat-{mat_name}" name="bsdf"/>')
        lines2.append('    <boolean name="face_normals" value="true"/>')
        lines2.append('  </shape>')

lines2.append('')
lines2.append('</scene>')

scene2_xml = os.path.join(SCENE_DIR, 'scene_sionna2.xml')
with open(scene2_xml, 'w') as f:
    f.write('\n'.join(lines2))

print(f'Wrote: {scene2_xml}')
print(f'  Materials : {len(used_mats2)}')
total2 = 1 + sum(len(v) for v in mat_plys.values())
print(f'  Shapes    : {total2}  (1 terrain + {total2-1} building PLYs)')
print(f'  Format    : Sionna 2.0 / Mitsuba 2.1.0  (twosided diffuse + face_normals)')
